# Saját model workflow

This notebook runs the custom `sajat` model by calling the Python modules in this folder.

It will:
- Train the model on `obesity_challenge_1.h5ad`.
- Run inference on a local test split.
- Save `prediction.h5ad` and `predict_program_proportion.csv` under `prediction/`.



In [1]:
!pip install scanpy

In [7]:
import scanpy
import pandas as pd


In [2]:
import os

DATA_DIR = "/home/efiareg/OneDrive/challenges/obesity/data"
MODEL_DIR = "/home/efiareg/OneDrive/challenges/obesity/sajat/resources"
PREDICTION_DIR = "/home/efiareg/OneDrive/challenges/obesity/sajat/prediction"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PREDICTION_DIR, exist_ok=True)

In [3]:
# Train the sajat model on the full training AnnData
from model import train, infer
train(
    data_directory_path=DATA_DIR,
    model_directory_path=MODEL_DIR,
)

print("Training finished. Artifacts saved in:", MODEL_DIR)



Training finished. Artifacts saved in: /home/efiareg/OneDrive/challenges/obesity/sajat/resources


In [8]:
# Run inference on a local test set and write predictions

# Local test perturbations from obesity_challenge_1_local_gtruth.h5ad
gtruth = scanpy.read_h5ad(os.path.join(DATA_DIR, "obesity_challenge_1_local_gtruth.h5ad"), backed="r")
predict_perturbations = gtruth.obs["gene"].cat.categories.tolist()

# Genes to predict from the bundle
genes_to_predict = pd.read_csv(os.path.join(DATA_DIR, "genes_to_predict.txt"), header=None)[0].tolist()

infer(
    data_directory_path=DATA_DIR,
    prediction_directory_path=PREDICTION_DIR,
    prediction_h5ad_file_path=os.path.join(PREDICTION_DIR, "prediction.h5ad"),
    program_proportion_csv_file_path=os.path.join(PREDICTION_DIR, "predict_program_proportion.csv"),
    model_directory_path=MODEL_DIR,
    predict_perturbations=predict_perturbations,
    genes_to_predict=genes_to_predict,
)

print("Inference finished. Predictions written under:", PREDICTION_DIR)



Inference finished. Predictions written under: /home/efiareg/OneDrive/challenges/obesity/sajat/prediction


In [9]:
# Quick preview of the generated files

prediction = scanpy.read_h5ad(os.path.join(PREDICTION_DIR, "prediction.h5ad"), backed="r")
print(prediction)

predicted_proportion = pd.read_csv(os.path.join(PREDICTION_DIR, "predict_program_proportion.csv"))
print(predicted_proportion.head())



AnnData object with n_obs × n_vars = 500 × 10237 backed at '/home/efiareg/OneDrive/challenges/obesity/sajat/prediction/prediction.h5ad'
    obs: 'gene'
     gene  pre_adipo     adipo      lipo     other
0    CHD4   0.350258  0.250367  0.073408  0.325967
1   FOXC1   0.350258  0.250367  0.073408  0.325967
2    SOX6   0.350258  0.250367  0.073408  0.325967
3   TRIM5   0.350258  0.250367  0.073408  0.325967
4  ZBTB20   0.350258  0.250367  0.073408  0.325967
